In [160]:
import re
import json
import pandas as pd

In [168]:
gold_standard_dataset = pd.read_csv('gold_standard_dataset.csv')

contains_fields = gold_standard_dataset["field"].str.contains(
        r"number_of_prior_pregnancies|time_seen_units|time_birth_units|\
        systolic_blood_pressure|diastolic_blood_pressure|rbs_measured|\
        given_bilirubin|primary_admission_diagnosis|secondary_admission_diagnosis",
    regex=True, na=False
    )
gold_standard_dataset = gold_standard_dataset[~contains_fields]

gold_standard_dataset = gold_standard_dataset.rename(columns={'value': 'gold_value'})
gold_standard_dataset.shape

(359797, 6)

In [169]:
gold_standard_dataset.head()

,hospital,patient_id,form_type,page,field,gold_value
0,76,76000172,NAR,2,appearance,well
1,76,76000172,NAR,2,capillary_refill_in_seconds,2
2,76,76000172,NAR,2,chest_indrawing,TRUE
3,76,76000172,NAR,2,given_bcg,NaN
4,76,76000172,NAR,2,given_bilirubin,NaN


In [170]:
list(gold_standard_dataset.gold_value.unique())

['well',
 '2',
 'TRUE',
 nan,
 'FALSE',
 'none',
 'normal',
 'reduced',
 'clean',
 '0',
 '4',
 '1',
 '24-03-2025',
 '700',
 'unknown',
 'facility',
 'svd',
 '27',
 '16',
 '85',
 '44',
 '39',
 'female',
 '34.8',
 '08:30',
 '790',
 '6',
 '8',
 '9',
 '06',
 '08-02-2025',
 '3600',
 '33',
 'us',
 'emergency',
 '32',
 '50',
 '21',
 '96',
 '145',
 '72',
 '35.8',
 '10:50',
 '11:30',
 '3250',
 'mild',
 '05',
 'referral',
 '15-01-2025',
 '1300',
 'o',
 '22-03-2025',
 '15-06-2024',
 '30',
 '3',
 '098',
 'positive',
 '36.4',
 '7',
 'postnatal',
 '22-04-2025',
 '1125',
 '08-07-2025',
 '01-10-2024',
 '29',
 '18',
 'male',
 '5',
 '00',
 '06-01-2025',
 '2400',
 'a',
 '07-02-2025',
 '35',
 'lmp',
 '17',
 '134',
 'lt18',
 '36.9',
 '18:20',
 '20:20',
 '2285',
 'sick',
 'weak',
 '02-01-2025',
 '2015',
 '05-02-2025',
 'cs',
 '36',
 '31',
 '56',
 '131',
 '38',
 '03:20',
 '01',
 '25-03-2025',
 '2800',
 '13-03-2025',
 '41',
 '28',
 '98',
 '138',
 '58',
 '01:20',
 '05:44',
 '2655',
 'severe',
 'pustules',
 '04

In [171]:
# db.webui_form_processor_stats.findOne({ image_filename: "ITF_40000133_page_1.png" })

gold_standard_dataset[
(gold_standard_dataset["field"]=="pulse_oximetry") & \
(gold_standard_dataset["form_type"]=="ITF") & \
(gold_standard_dataset["patient_id"]==40000133)]

,hospital,patient_id,form_type,page,field,gold_value
6435,40,40000133,ITF,1,pulse_oximetry,69


In [172]:
page_fields = gold_standard_dataset.groupby(["form_type", "page"])['field'].agg(lambda x: sorted(x.unique()))
gold_data_fields = {
    f"{form_type}_{page}": fields
    for (form_type, page), fields in page_fields.items()
}
gold_data_fields

{'ITF_1': ['abnormal_placenta',
  'anc_visits',
  'antenatal_steroids',
  'apgar_10m',
  'apgar_1m',
  'apgar_5m',
  'attended_anc',
  'baby_age',
  'baby_from',
  'birth_date',
  'birth_weight',
  'blood_group',
  'chest_compressions',
  'date_estimated_delivery_date',
  'date_last_menstrual_period',
  'delivery_type',
  'fetal_distress',
  'gestation_in_weeks',
  'given_bcg',
  'given_chlorhexidine',
  'given_teo',
  'given_vitamin_k',
  'gravida',
  'had_cs',
  'has_fever',
  'maternal_status',
  'multiple_pregnancy',
  'mum_age_in_years',
  'mum_had_antepartum_haemorrhage',
  'mum_had_diabetes',
  'mum_had_eclampsia',
  'mum_had_hep_b',
  'mum_had_hypertension_in_pregnancy',
  'mum_had_pre_eclampsia',
  'mum_had_vdrl',
  'mum_has_anc_ultrasound',
  'mum_on_arvs',
  'mum_pmtct_status',
  'mum_treated_for_tb',
  'parity_abortions',
  'parity_live',
  'passed_meconium',
  'placenta_complete',
  'prescribed_antibiotics',
  'prescribed_cpap',
  'prescribed_opv',
  'prescribed_oxygen',
 

In [173]:
with open("mongodb_key_structure.json", "r", encoding="utf-8") as f:
    llm_fields = json.load(f)

llm_fields

{'NAR_2': {'general_examination': ['abdominal_distension_present',
   "air_entry_in_baby's_lungs",
   "baby's_cry",
   "baby's_muscle_tone",
   "baby's_skin_condition",
   'baby_has_crackles',
   'baby_has_grunting',
   'baby_is_irritable',
   'capillary_refill_time_at_sternal_site',
   'condition_of_the_umbilicus',
   "cyanosis_present_in_baby's_central_body",
   'does_baby_have_bulging_fontanelle',
   'general_appearance_of_baby',
   'indrawing_of_lower_chest',
   'level_of_jaundice',
   'pallor_or_anaemia_present_in_baby',
   'presence_of_heart_murmur',
   'retraction_of_intercostal_muscles',
   'retraction_of_xiphoid_process'],
  'further_examination': ['birth_defects_present_in_baby',
   'birth_injury_or_other_abnormalities',
   'cleft_lip_or_palate_present',
   'further_examination_findings_for_respiratory_cardiovascular_git_gu_skin_and_birth_trauma',
   'hydrocephalus_present',
   'limb_abnormalities_present',
   'major_gastrointestinal_abnormality',
   'microcephaly_present',
 

In [174]:
llm_gold_key_mapper = {}
llm_gold_key_mapper["ITF_1"]={
  'abnormal_placenta':'placental_abnormalities',
  'anc_visits':'number_of_anc_visits_attended',
  'antenatal_steroids':'corticosteroids_given',
  'apgar_10m':'apgar_score_at_10_minutes',
  'apgar_1m': 'apgar_score_at_1_minute',
  'apgar_5m':'apgar_score_at_5_minutes',
  'attended_anc':'mother_attended_antenatal_care',
  'baby_age':'neonatal_age',
  'baby_from':'location_baby_originated_from',
  'birth_date':"baby's_date_of_birth",
  'birth_weight':'birth_weight_in_grams',
  'blood_group':"mother's_blood_group",
  'chest_compressions':'chest_compressions_performed',
  'date_estimated_delivery_date':'expected_date_of_delivery',
  'date_last_menstrual_period': 'last_menstrual_period',
  'delivery_type':'mode_of_delivery',
  'fetal_distress':'fetal_distress_during_labour',
  'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
  'given_bcg':'bcg_vaccine',
  'given_chlorhexidine':'chlorhexidine_cord_care',
  'given_teo': 'thermal_care', # mis-labelled in original dataset
  'given_vitamin_k':'vitamin_k_prophylaxis_given',
  'gravida':'total_number_of_pregnancies',
  'had_cs':'type_of_caesarean_section',
  'has_fever':'maternal_fever_present',
  'maternal_status':"mother's_current_location",
  'multiple_pregnancy':'multiple_pregnancy',
  'mum_age_in_years': "mother's_age_in_years",
  'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
  'mum_had_diabetes':'mother_has_diabetes',
  'mum_had_eclampsia':'eclampsia',
  'mum_had_hep_b':'hepatitis_b_vaccine',
  'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
  'mum_had_pre_eclampsia':'pre-eclampsia',
  'mum_had_vdrl':'syphilis_screening_vdrl_test',
  'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
  'mum_on_arvs': 'mother_on_antiretroviral_therapy',
  'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
  'mum_treated_for_tb':'mother_on_tb_treatment',
  'parity_abortions':'number_of_stillbirths/deaths',
  'parity_live':'number_of_live_births',
  'pulse_oximetry':'oxygen_saturation',
  'passed_meconium':'meconium-stained_amniotic_fluid',
  'placenta_complete':'placenta_completely_delivered',
  'prescribed_antibiotics':'mother_on_antibiotics',
  'prescribed_cpap':'cpap_support',
  'prescribed_opv':'oral_polio_vaccine',
  'prescribed_oxygen':'supplemental_oxygen',
  'pulse_rate':'heart_rate_beats_per_minute',
  'rapture_of_membrane':'rupture_of_membranes_timing_eg_>18h',
  'respiratory_rate':'respiratory_rate',
  'rhesus':'rhesus_factor_status',
  'sex':"baby's_sex",
  'temparature':'temperature_in_°c',
  'was_resuscitated':'bag_and_mask_ventilation_given',
  'weight':'current_weight_in_grams',
}

In [175]:
llm_fields['NAR_1']

{'infant_details': ['apgar_score_at_10_minutes',
  'apgar_score_at_1_minute',
  'apgar_score_at_5_minutes',
  "baby's_age_in_days",
  "baby's_date_of_birth",
  "baby's_sex",
  "baby's_time_of_birth",
  'bag_and_mask_ventilation_given',
  'date_infant_admitted',
  'gestational_age_at_delivery_in_weeks',
  'gestational_age_calculated_from',
  'if_baby_is_born_outside_facility',
  'is_baby_born_outside_facility',
  'mode_of_delivery',
  'multiple_deliveries',
  'number_of_fetuses_in_multiple_pregnancy',
  'rupture_of_membranes_timing_in_hours',
  'time_baby_seen',
  'type_of_caesarean_section'],
 'mother_details': ['antenatal_ultrasound_performed',
  'antepartum_hemorrhage',
  'expected_date_of_delivery',
  'hiv_status_prevention_of_mother-to-child_transmission',
  'hypertension_in_pregnancy',
  "mother's_age_in_years",
  "mother's_blood_group",
  'mother_given_hbig_treatment',
  'mother_had_hepatitis_b',
  'mother_had_prolonged_labour',
  'mother_has_diabetes',
  'mother_on_arvs',
  'num

In [176]:
llm_gold_key_mapper["NAR_1"]={
    'date':'date_infant_admitted',
    'anc_visits':'number_of_anc_visits_attended',
    'apgar_10m':'apgar_score_at_10_minutes',
    'apgar_1m':'apgar_score_at_1_minute',
    'apgar_5m':'apgar_score_at_5_minutes',
    'birth_date':"baby's_date_of_birth",
    'birth_weight':'birth_weight_in_grams',
    'blood_group':"mother's_blood_group",

    'baby_age_in_days':"baby's_age_in_days",
    'born_before_arrival':'is_baby_born_outside_facility',
    'mum_given_HBIG_treatment':'mother_given_hbig_treatment',
    'mum_on_arvs':'mother_on_arvs',
    'head_circumference':'head_circumference_in_cm',
    'length':'length_in_cm',
    'parity_abortions':'number_of_stillbirths/deaths',
    'parity_live':'number_of_live_births',
    'mum_had_hepatitis_b':'mother_had_hepatitis_b',
    'born_where':'if_baby_is_born_outside_facility',
    'date_estimated_delivery_date':'expected_date_of_delivery',
    'delivery_type':'mode_of_delivery',
    'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
    'gestation_type':'gestational_age_calculated_from',
    'given_anti_D_medication':'rhesus_anti-d_given',
    'had_cs':'type_of_caesarean_section',
    'has_apnoea':'baby_has_apnoea',
    'has_convulsions':'baby_has_convulsions',
    'has_diarhoea':'baby_has_bloody_stool',
    'has_difficulty_breathing':'baby_has_difficulty_breathing',
    'has_difficulty_feeding':'baby_has_difficulty_feeding',
    'has_fever':'baby_has_fever_present',
    'has_vomiting':'baby_has_bilious_vomiting',
    'is_floppy':'baby_is_floppy',
    'is_multiple_delivery':'multiple_deliveries',
    'multiple_delivery_num':'number_of_fetuses_in_multiple_pregnancy',
    'mum_age_in_years': "mother's_age_in_years",
    'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
    'mum_had_diabetes':'mother_has_diabetes',
    'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
    'mum_had_vdrl':'syphilis_screening_vdrl_test',
    'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
    'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
    'passed_meconium':'baby_passed_meconium_stool',
    'passed_urine':'baby_passed_urine',
    'prolonged_labour':'mother_had_prolonged_labour',
    'pulse_oximetry':'oxygen_saturation',
    'pulse_rate':'heart_rate_beats_per_minute',
    'rapture_of_membrane':'rupture_of_membranes_timing_in_hours',
    'respiratory_rate':'respiratory_rate',
    'rhesus':'rhesus_factor_status',
    'sex':"baby's_sex",
    'temparature':'temperature_in_°c',
    'time_birth':"baby's_time_of_birth",
    'time_seen':'time_baby_seen',
    'was_resuscitated':'bag_and_mask_ventilation_given',
    'weight':'current_weight_in_grams'
}

In [177]:
llm_gold_key_mapper["NAR_2"]={
    'appearance':'general_appearance_of_baby',
    'capillary_refill_in_seconds':'capillary_refill_time_at_sternal_site',
    'chest_indrawing':'indrawing_of_lower_chest',
    'cry':"baby's_cry",
    'given_bcg':'bcg_vaccination_given',
    'given_chlorhexidine':'chlorhexidine_given_for_cord_care',
    'given_prophylaxis_pmtct':'prophylaxis_for_prevention_of_mother-to-child_transmission',
    'given_vitamin_k':'vitamin_k_and_topical_eye_ointment_given',
    'has_birth_defects':'birth_defects_present_in_baby',
    'has_bulging_fontanelle':'does_baby_have_bulging_fontanelle',
    'has_central_cyanosis':"cyanosis_present_in_baby's_central_body",
    'has_crackles':'baby_has_crackles',
    'has_good_air_entry':"air_entry_in_baby's_lungs",
    'has_grunting':'baby_has_grunting',
    'has_murmur':'presence_of_heart_murmur',
    'intercostal_retraction':'retraction_of_intercostal_muscles',
    'is_distended':'abdominal_distension_present',
    'is_irritable':'baby_is_irritable',
    'jaundice':'level_of_jaundice',
    'pallor':'pallor_or_anaemia_present_in_baby',
    'prescribed_antibiotics':'antibiotic_therapy_given',
    'prescribed_caffeine_citrate':'caffeine_citrate_given_for_apnoea',
    'prescribed_cpap':'cpap_therapy_given',
    'prescribed_feeds':'feeding/nutrition_support',
    'prescribed_incubator':'incubator_or_warm_environment_provided',
    'prescribed_iv_fluids':'intravenous_fluids_given',
    'prescribed_kmc':'kangaroo_mother_care_provided',
    'prescribed_opv':'opv_polio_vaccination_given',
    'prescribed_oxygen':'oxygen_therapy_given',
    'prescribed_phototherapy':'phototherapy_given_for_jaundice',
    'prescribed_surfactant':'surfactant_therapy_given',
    'prescribed_transfusion':'blood_transfusion_given',
    'skin':"baby's_skin_condition",
    'tone':"baby's_muscle_tone",
    'umbilicus':'condition_of_the_umbilicus',
    'xiphoid_retraction':'retraction_of_xiphoid_process'
}

In [178]:
llm_gold_key_mapper

{'ITF_1': {'abnormal_placenta': 'placental_abnormalities',
  'anc_visits': 'number_of_anc_visits_attended',
  'antenatal_steroids': 'corticosteroids_given',
  'apgar_10m': 'apgar_score_at_10_minutes',
  'apgar_1m': 'apgar_score_at_1_minute',
  'apgar_5m': 'apgar_score_at_5_minutes',
  'attended_anc': 'mother_attended_antenatal_care',
  'baby_age': 'neonatal_age',
  'baby_from': 'location_baby_originated_from',
  'birth_date': "baby's_date_of_birth",
  'birth_weight': 'birth_weight_in_grams',
  'blood_group': "mother's_blood_group",
  'chest_compressions': 'chest_compressions_performed',
  'date_estimated_delivery_date': 'expected_date_of_delivery',
  'date_last_menstrual_period': 'last_menstrual_period',
  'delivery_type': 'mode_of_delivery',
  'fetal_distress': 'fetal_distress_during_labour',
  'gestation_in_weeks': 'gestational_age_at_delivery_in_weeks',
  'given_bcg': 'bcg_vaccine',
  'given_chlorhexidine': 'chlorhexidine_cord_care',
  'given_teo': 'thermal_care',
  'given_vitamin_k

In [179]:
eval_keys = list(llm_gold_key_mapper["ITF_1"].keys()) + list(llm_gold_key_mapper["NAR_1"].keys()) + list(llm_gold_key_mapper["NAR_2"].keys())
eval_keys = list(set(eval_keys))

gold_standard_dataset = gold_standard_dataset[
    gold_standard_dataset['field'].isin(eval_keys)
]
gold_standard_dataset.shape

(354347, 6)

In [180]:
def restructure_dataset(data: dict, key_mapper: dict) -> dict:
    """
    Restructure a JSON-loaded dataset keyed by filenames like
    'NAR_52000552_page_2.png', adding parsed metadata fields
    (form_type, hospital, patient_id, page), and nesting the
    field/value pairs under a '<form_type>_<page>' key — filtered
    and renamed according to key_mapper.

    key_mapper shape: { "NAR_2": {"skin": "baby's_skin_condition", ...}, ... }
    i.e. group_key -> {short_name: descriptive_name}
    """
    result = {}

    # Matches: <form_type>_<patient_id>_page_<page>.png
    pattern = re.compile(r"^([A-Za-z]+)_(\d+)_page_(\d+)\.png$")

    # Pre-invert each group's mapper: descriptive_name -> short_name
    inverted_mappers = {
        group_key: {desc: short for short, desc in mapping.items()}
        for group_key, mapping in key_mapper.items()
    }

    unmapped_groups = set()

    for filename, fields in data.items():
        match = pattern.match(filename)
        if not match:
            print(f"Skipping unrecognized filename format: {filename}")
            continue

        form_type, patient_id, page = match.groups()
        hospital = patient_id[:2]
        group_key = f"{form_type}_{page}"

        desc_to_short = inverted_mappers.get(group_key)
        if desc_to_short is None:
            unmapped_groups.add(group_key)
            filtered_fields = {}
        else:
            # Keep only fields present in the mapper, renamed to short key
            filtered_fields = {
                desc_to_short[desc_name]: value
                for desc_name, value in fields.items()
                if desc_name in desc_to_short
            }

        result[filename] = {
            "form_type": form_type,
            "hospital": hospital,
            "patient_id": patient_id,
            "page": page,
            group_key: filtered_fields,
        }

    if unmapped_groups:
        print(f"No key mapper found for group(s): {sorted(unmapped_groups)} — fields left empty for these.")

    return result

llm_dataset = json.load(open('llm_dataset.json'))
restructured_llm_dataset = restructure_dataset(llm_dataset, llm_gold_key_mapper)

In [181]:
restructured_llm_dataset["NAR_52000552_page_1.png"]

{'form_type': 'NAR',
 'hospital': '52',
 'patient_id': '52000552',
 'page': '1',
 'NAR_1': {'date': '2025-03-29',
  'time_seen': '02:32',
  'sex': 'Female',
  'birth_date': '2025-03-29',
  'time_birth': '02:44',
  'gestation_in_weeks': 36,
  'baby_age_in_days': 1,
  'gestation_type': 'U/S',
  'apgar_1m': 8,
  'apgar_5m': 9,
  'apgar_10m': 10,
  'delivery_type': 'SVD',
  'was_resuscitated': False,
  'rapture_of_membrane': '<18 hrs',
  'is_multiple_delivery': False,
  'born_before_arrival': False,
  'born_where': 'Home/roadside',
  'mum_age_in_years': 22,
  'parity_live': 0,
  'parity_abortions': 0,
  'date_estimated_delivery_date': '2026-01-27',
  'anc_visits': 1,
  'mum_has_anc_ultrasound': True,
  'blood_group': 'A',
  'rhesus': 'Positive',
  'given_anti_D_medication': True,
  'mum_had_vdrl': 'Negative',
  'mum_pmtct_status': 'Unknown',
  'mum_had_hypertension_in_pregnancy': True,
  'mum_given_HBIG_treatment': 'Unknown',
  'mum_on_arvs': 'Unknown',
  'mum_had_antepartum_haemorrhage': 

In [182]:
def to_long_dataframe(restructured: dict) -> pd.DataFrame:
    """
    Convert the restructured dataset (keyed by filename, with metadata
    fields + a nested 'form_type_page' field/value dict) into a long-format
    pandas DataFrame with columns:
    ['hospital', 'patient_id', 'form_type', 'page', 'field', 'value'].
    """
    rows = []

    for filename, record in restructured.items():
        form_type = record["form_type"]
        page = record["page"]
        hospital = record["hospital"]
        patient_id = record["patient_id"]

        group_key = f"{form_type}_{page}"
        fields = record.get(group_key, {})

        for field, value in fields.items():
            rows.append({
                "hospital": hospital,
                "patient_id": patient_id,
                "form_type": form_type,
                "page": page,
                "field": field,
                "llm_value": value,
            })

    return pd.DataFrame(rows, columns=["hospital", "patient_id", "form_type", "page", "field", "llm_value"])

llm_dataset_df = to_long_dataframe(restructured_llm_dataset)

cols_to_convert = ['hospital', 'patient_id', 'page']

# Convert columns to int64
llm_dataset_df[cols_to_convert] = llm_dataset_df[cols_to_convert].astype('int64')

# Convert to string
llm_dataset_df[['llm_value']] = llm_dataset_df[['llm_value']].astype('str')

# Keep records in gold dataset are those also in the llm dataset
gold_standard_dataset['key'] = gold_standard_dataset['form_type'] + "_" + gold_standard_dataset['patient_id'].astype(str) + "_" + gold_standard_dataset['page'].astype(str)
llm_dataset_df['key'] = llm_dataset_df['form_type'] + "_" + llm_dataset_df['patient_id'].astype(str) + "_" + llm_dataset_df['page'].astype(str)

gold_standard_df = gold_standard_dataset[
    gold_standard_dataset['key'].isin(llm_dataset_df['key'])
]

gold_standard_df = gold_standard_df.drop(columns=['key'])
llm_dataset_df = llm_dataset_df.drop(columns=['key'])

eval_dataset = pd.merge(
    gold_standard_df,
    llm_dataset_df,
    on=["hospital", "patient_id", "form_type", "page", "field",],
    how='left'
)

eval_dataset.dtypes

hospital      int64
patient_id    int64
form_type       str
page          int64
field           str
gold_value      str
llm_value       str
dtype: object

In [183]:
llm_dataset_df.shape

(273103, 6)

In [184]:
gold_standard_df.shape

(346373, 6)

In [185]:
eval_dataset.shape

(346373, 7)

In [186]:
eval_dataset

,hospital,patient_id,form_type,page,field,gold_value,llm_value
0,76,76000172,NAR,2,appearance,well,Well
1,76,76000172,NAR,2,capillary_refill_in_seconds,2,2 seconds
2,76,76000172,NAR,2,chest_indrawing,TRUE,True
3,76,76000172,NAR,2,given_bcg,NaN,NaN
4,76,76000172,NAR,2,given_chlorhexidine,TRUE,True
...,...,...,...,...,...,...,...
346368,72,72000885,ITF,1,rhesus,positive,Positive
346369,72,72000885,ITF,1,sex,male,Male
346370,72,72000885,ITF,1,temparature,56.9,36.9
346371,72,72000885,ITF,1,was_resuscitated,FALSE,False


In [187]:
eval_dataset.to_csv("eval_dataset.csv", index=False)